In [ ]:
import pandas as pd
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

##EXERCISE 1

Трансформирајте го податочното множество во формат соодветен за
класификација на текст. Користејќи го моделот LLaMA-2 со квантизација со 4bits со
техниката few-shot prompting за секој примерок од податочното множество
одредете дали примерокот содржи позитивен или негативен сентимент. Испробајте
со користење различен број на примероци (n = 0, 1, 5, 10).
Добиените предвидувања евалуирајте ги со метриките: точност
(accuracy_score), прецизност (precision_score), одзив (recall_score) и F1-
мерка (f1_score).

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/NLP2025/yelp_parallel/yelp_parallel/test_en_parallel.txt', sep='\t', header=None)
df.columns = ["Negative", "Positive"]
df = df.iloc[1:].reset_index(drop=True)
df.head(6)

In [ ]:
data = (
    pd.concat(
        [
            df["Negative"].to_frame("Text").assign(Label="Negative"),
            df["Positive"].to_frame("Text").assign(Label="Positive")
        ],
        ignore_index=True
    )
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

data.head(6)

In [ ]:
def build_prompt(text, examples=None):
    prompt = (
        "Classify the sentiment of the following text as Positive or Negative.\n"
    )

    if examples:
        for ex_text, ex_label in examples:
            prompt += f"Text: {ex_text}\nSentiment: {ex_label}\n\n"

    prompt += f"Text: {text}\nSentiment:"
    return prompt

#Testing prompt
test_text = data.loc[0, "Text"]
prompt = build_prompt(test_text)
print(prompt)

In [ ]:
shot = [(data.loc[1, "Text"], data.loc[1, "Label"])]

prompt_one_shot = build_prompt(test_text, examples=shot)
print(prompt_one_shot)


In [ ]:
from huggingface_hub import login
login()

In [ ]:
!pip install -U bitsandbytes accelerate transformers

In [ ]:
model_id = "meta-llama/Llama-2-7b-chat-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto'
)

In [ ]:
print(model)

In [ ]:
def predict_sentiment(text, examples=None, max_new_tokens=5):
    prompt = build_prompt(text, examples)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    prediction = response.split("Sentiment:")[-1].strip()

    if "Positive" in prediction:
        return "Positive"
    elif "Negative" in prediction:
        return "Negative"
    else:
        return "Negative"  # fallback


In [ ]:
import random

def get_few_shot_examples_random(data, n, exclude_idx=None):
    if n == 0:
        return None
    if exclude_idx is not None:
        data = data.drop(index=exclude_idx)

    pos_samples = data[data["Label"] == "Positive"].sample(n=n // 2 + n % 2, random_state=42)
    neg_samples = data[data["Label"] == "Negative"].sample(n=n // 2, random_state=42)

    examples = [(row["Text"], row["Label"]) for _, row in pd.concat([pos_samples, neg_samples]).iterrows()]
    random.shuffle(examples)
    return examples


In [ ]:
test_idx = 5
few_shots = get_few_shot_examples_random(data, n=1, exclude_idx=test_idx)
print(few_shots)

In [ ]:
#one shot testing
print("Text:", data.loc[test_idx, "Text"])
print("TRUE:", data.loc[test_idx, "Label"])
print("PRED:", predict_sentiment(data.loc[test_idx, "Text"], examples=few_shots,max_new_tokens=2))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(data, n_shots, limit=100):
    y_true = []
    y_pred = []

    for idx, row in data.head(limit).iterrows():
        examples = get_few_shot_examples_random(data, n_shots, exclude_idx=idx)
        pred = predict_sentiment(row["Text"], examples)

        y_pred.append(pred)
        y_true.append(row["Label"])

    # Compute metrics
    return {
        "shots": n_shots,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label="Positive"),
        "recall": recall_score(y_true, y_pred, pos_label="Positive"),
        "f1": f1_score(y_true, y_pred, pos_label="Positive")
    }


In [ ]:
results = []

for n in [0, 1, 5, 10]:
    metrics = evaluate_model(data, n_shots=n, limit=100)
    results.append(metrics)

results_df = pd.DataFrame(results)
results_df

The model performs well even in zero-shot settings (F1 ≈ 0.92), showing strong inherent sentiment understanding. One few-shot example slightly reduces performance, likely due to bias from a single example. With five few-shot examples, the model achieves the highest precision, indicating improved generalization. Ten examples increase recall, catching nearly all positives, though precision drops slightly. Overall, few-shot prompting improves F1, with five to ten examples providing the best balance, while zero-shot remains a solid baseline.

LLaMA-2 is not inherently superior to the models used in previous lab exercises, however, it provides greater flexibility through few-shot prompting without requiring additional training. While the earlier models are faster and more efficient for specific, well-defined tasks, LLaMA-2 excels in adaptability and can be applied to a wider range of tasks with minimal effort.

## EXERCISE 2


Користејќи го моделот LLaMA-2 со квантизација со 4bits со техниката few-shot
prompting за секој примерок од податочното множествo што содржи негативен
сентимент генерирајте реченица што содржи позитивен сентимент. Испробајте со
користење различен број на примероци (n = 0, 1, 5, 10).
Добиените предвидувања евалуирајте ги со метриките: BLEU и BERTScore.


In [ ]:
def get_few_shot_positive_examples(data, n, exclude_idx=None):
    if n == 0:
        return None

    pos_data = data[data["Label"] == "Positive"]

    if exclude_idx is not None:
        pos_data = pos_data.drop(index=exclude_idx)

    examples = pos_data.sample(n=n, random_state=42)
    examples_list = [(row["Text"], row["Label"]) for _, row in examples.iterrows()]
    random.shuffle(examples_list)

    return examples_list

In [ ]:
def build_rewrite_prompt(negative_text, examples=None):
    prompt = "Rewrite the following negative sentence into a positive one.\n"

    if examples:
        for ex_text, _ in examples:
            prompt += f"Example: {ex_text} -> {ex_text} (positive)\n"

    prompt += f"Input: {negative_text}\nOutput (positive):"
    return prompt


In [ ]:
def generate_positive(negative_text, examples=None, max_length=100):
    prompt = build_rewrite_prompt(negative_text, examples)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=max_length)
    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return output_text.split("Output (positive):")[-1].strip()

In [ ]:
!pip install bert-score nltk

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "meta-llama/Llama-2-7b-chat-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

def generate_positive(negative_text, examples=None, max_length=100):
    prompt = build_rewrite_prompt(negative_text, examples)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=max_length)
    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return output_text.split("Output (positive):")[-1].strip()


In [ ]:
import bert_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def evaluate_rewriting(df_eval, data_examples, n_shots, limit=50):
    references = []
    predictions = []
    smoothing = SmoothingFunction().method1

    for idx, row in df_eval.head(limit).iterrows():
        neg_text = row["Negative"]
        print(neg_text)
        pos_ref = row["Positive"]
        print(pos_ref)

        examples = get_few_shot_positive_examples(data_examples, n_shots, exclude_idx=None)
        pred = generate_positive(neg_text, examples)

        if not pred.strip():
            pred = "<empty>"

        predictions.append(pred)
        references.append(pos_ref)

    # Compute BLEU
    bleu_scores = [
        sentence_bleu([ref.split()], pred.split(), smoothing_function=smoothing)
        for pred, ref in zip(predictions, references)
    ]
    avg_bleu = sum(bleu_scores) / len(bleu_scores)

    # Compute BERTScore
    P, R, F1 = bert_score.score(predictions, references, lang="en", rescale_with_baseline=True)
    avg_bertscore = F1.mean().item()

    return {
        "shots": n_shots,
        "BLEU": avg_bleu,
        "BERTScore": avg_bertscore
    }


In [ ]:
results = []
for n in [0, 1, 5, 10]:
    res = evaluate_rewriting(df, data, n_shots=n, limit=40)  # df for evaluation, data for few-shot
    results.append(res)

results_df = pd.DataFrame(results)
results_df

Few-shot prompting with LLaMA-2 improves its ability to rewrite negative sentences into positive ones. BLEU scores remain low, reflecting lexical differences, but BERTScore steadily increases from 0.136 to 0.449 as more positive examples are provided, showing better semantic alignment. Most gains occur by five examples, indicating diminishing returns beyond that. Overall, few-shot prompting effectively guides LLaMA-2 to produce semantically meaningful positive rewrites